# Domain Adaptation for Sentiment Analysis
**Source domain:** IMDB movie reviews  
**Target domain:** Amazon product reviews  

## Pipeline
1. DAPT — continue BERT MLM pretraining on unlabelled Amazon text (Pool A)
2. Fine-tune on labelled IMDB
3. Pseudo-labeling on Pool B using the DAPT+IMDB model

## Baselines (all evaluated on Pool C)
| # | Baseline |
|---|----------|
| 1 | VADER (rule-based) |
| 2 | BERT + IMDB fine-tune |
| 3 | BERT + IMDB + small labelled Amazon (1000 samples) |
| 4 | BERT + IMDB + pseudo-labeling (no DAPT) |
| 5 | BERT + DAPT + IMDB fine-tune |
| 6 | BERT + DAPT + IMDB + pseudo-labeling (full pipeline) |

In [ ]:
!pip install transformers datasets evaluate scikit-learn pandas torch accelerate vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 9.4 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import evaluate
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForMaskedLM,
    DataCollatorWithPadding,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed
)
from torch.nn.functional import softmax
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [ ]:
MODEL_NAME       = "bert-base-uncased"
NUM_LABELS       = 2
MAX_LENGTH       = 256
SEED             = 42
PSEUDO_THRESHOLD = 0.9
SMALL_TARGET_SIZE = 1000   # labelled Amazon samples for Baseline 3

DAPT_OUTPUT_DIR          = "./bert_dapt"
IMDB_OUTPUT_DIR          = "./bert_imdb"
DAPT_IMDB_OUTPUT_DIR     = "./bert_dapt_imdb"
SMALL_TARGET_OUTPUT_DIR  = "./bert_imdb_small_target"
PSEUDO_OUTPUT_DIR        = "./bert_imdb_pseudo"
PSEUDO_DAPT_OUTPUT_DIR   = "./bert_dapt_imdb_pseudo"

set_seed(SEED)

## Data Loading & Splitting

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --- Amazon ---
amazon_df = pd.read_csv("/content/drive/MyDrive/amazon.csv", header=None)
amazon_df.columns = ["label", "title", "review"]
amazon_df["text"] = amazon_df["title"] + " " + amazon_df["review"]
amazon_df["label"] = amazon_df["label"].map({1: 0, 2: 1})  # 1-star -> 0 (neg), 2-star -> 1 (pos)
amazon_df = amazon_df[["text", "label"]].dropna()

# --- IMDB ---
imdb_df = pd.read_csv("/content/drive/MyDrive/IMDB.csv")
imdb_df = imdb_df.rename(columns={"review": "text", "sentiment": "label"})
imdb_df["label"] = imdb_df["label"].map({"negative": 0, "positive": 1})
imdb_df = imdb_df[["text", "label"]].dropna()

print(f"Amazon: {len(amazon_df)} samples | label dist: {amazon_df['label'].value_counts().to_dict()}")
print(f"IMDB:   {len(imdb_df)} samples | label dist: {imdb_df['label'].value_counts().to_dict()}")

Mounted at /content/drive
Amazon: 49999 samples | label dist: {1: 25027, 0: 24972}
IMDB:   50000 samples | label dist: {1: 25000, 0: 25000}


In [ ]:
# =========================
# Amazon pool splitting  (60 / 20 / 20, stratified)
# =========================

# Pool A (60%) — raw text for DAPT, labels stripped
pool_a_df, pool_bc_df = train_test_split(
    amazon_df, test_size=0.4,
    stratify=amazon_df["label"], random_state=SEED
)

# Pool B (20%) — unlabelled, for pseudo-labeling
# Pool C (20%) — held-out test set
pool_b_df, pool_c_df = train_test_split(
    pool_bc_df, test_size=0.5,
    stratify=pool_bc_df["label"], random_state=SEED
)

# Strip labels from Pool A (DAPT uses no label information)
pool_a_text_df = pool_a_df[["text"]].reset_index(drop=True)

# Strip labels from Pool B (pseudo-labeling starts from unlabelled text)
pool_b_unlabeled_df = pool_b_df[["text"]].reset_index(drop=True)

# Small labelled target set for Baseline 3 (sampled from Pool B, stratified)
# We sample from Pool B since Pool A has no labels and Pool C is held-out
small_target_df, _ = train_test_split(
    pool_b_df, train_size=SMALL_TARGET_SIZE,
    stratify=pool_b_df["label"], random_state=SEED
)

print(f"Pool A (DAPT):            {len(pool_a_df):>6} samples (labels stripped)")
print(f"Pool B (pseudo-labeling): {len(pool_b_df):>6} samples (unlabelled)")
print(f"Pool C (held-out test):   {len(pool_c_df):>6} samples")
print(f"Small labelled target:    {len(small_target_df):>6} samples (from Pool B)")

Pool A (DAPT):             29999 samples (labels stripped)
Pool B (pseudo-labeling):  10000 samples (unlabelled)
Pool C (held-out test):    10000 samples
Small labelled target:      1000 samples (from Pool B)


In [ ]:
# =========================
# IMDB train/validation split
# =========================
imdb_train_df, imdb_valid_df = train_test_split(
    imdb_df, test_size=0.2,
    stratify=imdb_df["label"], random_state=SEED
)

print(f"IMDB train: {len(imdb_train_df)} | IMDB valid: {len(imdb_valid_df)}")

IMDB train: 40000 | IMDB valid: 10000


## Tokenizer & Shared Utilities

In [ ]:
# =========================
# Tokenizer
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_clf(examples):
    """Tokenizer for classification tasks (has labels)."""
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LENGTH)

def tokenize_mlm(examples):
    """Tokenizer for MLM / DAPT (text only, no labels)."""
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LENGTH)

data_collator_clf = DataCollatorWithPadding(tokenizer=tokenizer)
data_collator_mlm = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# =========================
# Metrics
# =========================
accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1  = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    precision, recall, f1_per_class, _ = precision_recall_fscore_support(
        labels, preds, average=None, zero_division=0
    )

    metrics = {"accuracy": acc, "macro_f1": f1}
    for i in range(len(precision)):
        metrics[f"precision_class_{i}"] = precision[i]
        metrics[f"recall_class_{i}"]    = recall[i]
        metrics[f"f1_class_{i}"]        = f1_per_class[i]
    return metrics


def build_training_args(output_dir, lr=2e-5, epochs=3, train_batch=16, eval_batch=32):
    """Shared TrainingArguments factory for classification fine-tuning."""
    return TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=train_batch,
        per_device_eval_batch_size=eval_batch,
        num_train_epochs=epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED
    )


def evaluate_on_pool_c(trainer, pool_c_tok, label):
    """Evaluate a trainer on Pool C and return a results dict."""
    print(f"\n--- Evaluating: {label} ---")
    results = trainer.evaluate(eval_dataset=pool_c_tok)
    print(results)
    return {"baseline": label, **results}

In [ ]:
# =========================
# Tokenize shared datasets
# =========================

# IMDB
imdb_train_tok = Dataset.from_pandas(imdb_train_df.reset_index(drop=True)).map(tokenize_clf, batched=True)
imdb_valid_tok = Dataset.from_pandas(imdb_valid_df.reset_index(drop=True)).map(tokenize_clf, batched=True)

# Pool C (held-out test)
pool_c_tok = Dataset.from_pandas(pool_c_df.reset_index(drop=True)).map(tokenize_clf, batched=True)

# Pool A (DAPT — MLM, text only)
pool_a_tok = Dataset.from_pandas(pool_a_text_df).map(tokenize_mlm, batched=True)

# Pool B (unlabelled — for pseudo-labeling inference)
pool_b_tok = Dataset.from_pandas(pool_b_unlabeled_df).map(tokenize_mlm, batched=True)

# Small labelled Amazon (Baseline 3)
small_target_tok = Dataset.from_pandas(small_target_df.reset_index(drop=True)).map(tokenize_clf, batched=True)

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/29999 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

## Baseline 1 — VADER (Rule-Based)

In [ ]:
# =========================
# Baseline 1: VADER
# =========================
analyzer = SentimentIntensityAnalyzer()

def vader_predict(texts, threshold=0.05):
    """
    Compound score >= threshold -> positive (1)
    Compound score <  threshold -> negative (0)
    Threshold of 0.05 is the standard VADER recommendation.
    """
    preds = []
    for text in texts:
        score = analyzer.polarity_scores(str(text))["compound"]
        preds.append(1 if score >= threshold else 0)
    return np.array(preds)

vader_preds  = vader_predict(pool_c_df["text"].tolist())
vader_labels = pool_c_df["label"].values

vader_acc = accuracy_score(vader_labels, vader_preds)
vader_f1  = f1_score(vader_labels, vader_preds, average="macro")

print(f"VADER | Accuracy: {vader_acc:.4f} | Macro F1: {vader_f1:.4f}")

baseline_results = [
    {"baseline": "1. VADER", "eval_accuracy": vader_acc, "eval_macro_f1": vader_f1}
]

VADER | Accuracy: 0.7247 | Macro F1: 0.7128


## Baseline 2 — BERT + IMDB Fine-tune

In [ ]:
# =========================
# Baseline 2: BERT + IMDB
# =========================
imdb_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

imdb_trainer = Trainer(
    model=imdb_model,
    args=build_training_args(IMDB_OUTPUT_DIR, lr=2e-5, epochs=3),
    train_dataset=imdb_train_tok,
    eval_dataset=imdb_valid_tok,
    data_collator=data_collator_clf,
    compute_metrics=compute_metrics
)

print("===== Baseline 2: BERT + IMDB fine-tune =====")
imdb_trainer.train()
imdb_trainer.save_model(IMDB_OUTPUT_DIR)
tokenizer.save_pretrained(IMDB_OUTPUT_DIR)

baseline_results.append(evaluate_on_pool_c(imdb_trainer, pool_c_tok, "2. BERT + IMDB"))

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


===== Baseline 2: BERT + IMDB fine-tune =====


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Precision Class 0,Recall Class 0,F1 Class 0,Precision Class 1,Recall Class 1,F1 Class 1
1,0.260632,0.214634,0.924800,0.924790,0.934889,0.913200,0.923917,0.915168,0.936400,0.925662
2,0.152351,0.246747,0.926300,0.926267,0.908882,0.947600,0.927837,0.945268,0.905000,0.924696
3,0.081261,0.300481,0.932800,0.932796,0.939659,0.925000,0.932272,0.926152,0.940600,0.933320


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Evaluating: 2. BERT + IMDB ---


{'eval_loss': 0.30825042724609375, 'eval_accuracy': 0.9138, 'eval_macro_f1': 0.9135942298427167, 'eval_precision_class_0': 0.9574939118884215, 'eval_recall_class_0': 0.8658658658658659, 'eval_f1_class_0': 0.9093776282590412, 'eval_precision_class_1': 0.8778041218311143, 'eval_recall_class_1': 0.9616383616383616, 'eval_f1_class_1': 0.9178108314263921, 'eval_runtime': 37.33, 'eval_samples_per_second': 267.881, 'eval_steps_per_second': 8.385, 'epoch': 3.0}


## Baseline 3 — BERT + IMDB + Small Labelled Amazon (1000 samples)

In [ ]:
# =========================
# Baseline 3: BERT + IMDB + small labelled Amazon
# Start from the saved IMDB model, fine-tune further on 1000 labelled Amazon samples
# =========================
small_target_model = AutoModelForSequenceClassification.from_pretrained(IMDB_OUTPUT_DIR, num_labels=NUM_LABELS)

small_target_trainer = Trainer(
    model=small_target_model,
    args=build_training_args(SMALL_TARGET_OUTPUT_DIR, lr=1e-5, epochs=5, train_batch=8),
    train_dataset=small_target_tok,
    eval_dataset=pool_c_tok,   # use Pool C as validation monitor (not for selection)
    data_collator=data_collator_clf,
    compute_metrics=compute_metrics
)

print("===== Baseline 3: BERT + IMDB + small labelled Amazon =====")
small_target_trainer.train()
small_target_trainer.save_model(SMALL_TARGET_OUTPUT_DIR)

baseline_results.append(evaluate_on_pool_c(small_target_trainer, pool_c_tok, "3. BERT + IMDB + Small Labelled Amazon"))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

===== Baseline 3: BERT + IMDB + small labelled Amazon =====


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Precision Class 0,Recall Class 0,F1 Class 0,Precision Class 1,Recall Class 1,F1 Class 1
1,0.277300,0.206597,0.943400,0.943399,0.946742,0.939540,0.943127,0.940115,0.947253,0.943670
2,0.093974,0.249606,0.944000,0.943992,0.932514,0.957157,0.944675,0.956085,0.930869,0.943308
3,0.057233,0.287075,0.940000,0.939976,0.957336,0.920921,0.938776,0.923965,0.959041,0.941176
4,0.032811,0.283890,0.942500,0.942494,0.950653,0.933333,0.941913,0.934655,0.951648,0.943075
5,0.016529,0.301176,0.941500,0.941486,0.954452,0.927127,0.940591,0.929293,0.955844,0.942382


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Evaluating: 3. BERT + IMDB + Small Labelled Amazon ---


{'eval_loss': 0.2495659738779068, 'eval_accuracy': 0.944, 'eval_macro_f1': 0.9439919348386168, 'eval_precision_class_0': 0.9326829268292683, 'eval_recall_class_0': 0.9569569569569569, 'eval_f1_class_0': 0.9446640316205533, 'eval_precision_class_1': 0.9558974358974359, 'eval_recall_class_1': 0.9310689310689311, 'eval_f1_class_1': 0.9433198380566802, 'eval_runtime': 37.461, 'eval_samples_per_second': 266.944, 'eval_steps_per_second': 8.355, 'epoch': 5.0}


## Baseline 4 — BERT + IMDB + Pseudo-labeling (No DAPT)

In [ ]:
# =========================
# Pseudo-label generation utility
# =========================
def generate_pseudo_labels(trainer, unlabeled_tok, threshold=PSEUDO_THRESHOLD):
    """
    Run inference on an unlabelled tokenized dataset.
    Keep only samples where max softmax probability >= threshold.
    Returns a tokenized Dataset with pseudo labels attached.
    """
    outputs      = trainer.predict(unlabeled_tok)
    probs        = softmax(torch.tensor(outputs.predictions), dim=1)
    confidences, preds = torch.max(probs, dim=1)
    mask         = confidences >= threshold

    # Rebuild a text-only dataframe then re-tokenize
    kept_texts  = [unlabeled_tok[i]["text"] for i in range(len(unlabeled_tok)) if mask[i]]
    kept_labels = preds[mask].numpy()

    print(f"Pseudo-labeled samples kept: {len(kept_texts)} / {len(unlabeled_tok)} "
          f"(threshold={threshold})")

    pseudo_df  = pd.DataFrame({"text": kept_texts, "label": kept_labels})
    pseudo_ds  = Dataset.from_pandas(pseudo_df.reset_index(drop=True))
    pseudo_tok = pseudo_ds.map(tokenize_clf, batched=True)
    return pseudo_tok

In [ ]:
# =========================
# Baseline 4: BERT + IMDB + pseudo-labeling (no DAPT)
# Teacher: IMDB-only model (Baseline 2)
# =========================
pseudo_b4_tok = generate_pseudo_labels(imdb_trainer, pool_b_tok)

pseudo_b4_model = AutoModelForSequenceClassification.from_pretrained(IMDB_OUTPUT_DIR, num_labels=NUM_LABELS)

pseudo_b4_trainer = Trainer(
    model=pseudo_b4_model,
    args=build_training_args(PSEUDO_OUTPUT_DIR, lr=1e-5, epochs=3, train_batch=8),
    train_dataset=pseudo_b4_tok,
    eval_dataset=pool_c_tok,
    data_collator=data_collator_clf,
    compute_metrics=compute_metrics
)

print("===== Baseline 4: BERT + IMDB + Pseudo-labeling (no DAPT) =====")
pseudo_b4_trainer.train()
pseudo_b4_trainer.save_model(PSEUDO_OUTPUT_DIR)

baseline_results.append(evaluate_on_pool_c(pseudo_b4_trainer, pool_c_tok, "4. BERT + IMDB + Pseudo (no DAPT)"))

Pseudo-labeled samples kept: 9427 / 10000 (threshold=0.9)


Map:   0%|          | 0/9427 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

===== Baseline 4: BERT + IMDB + Pseudo-labeling (no DAPT) =====


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Precision Class 0,Recall Class 0,F1 Class 0,Precision Class 1,Recall Class 1,F1 Class 1
1,0.086640,0.429569,0.932100,0.932067,0.950898,0.911111,0.930580,0.914845,0.953047,0.933555
2,0.028549,0.589912,0.924200,0.924050,0.964481,0.880681,0.920678,0.890421,0.967632,0.927422
3,0.011819,0.588000,0.928600,0.928496,0.963011,0.891291,0.925764,0.899014,0.965834,0.931227


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Evaluating: 4. BERT + IMDB + Pseudo (no DAPT) ---


{'eval_loss': 0.42999330163002014, 'eval_accuracy': 0.9321, 'eval_macro_f1': 0.9320668207559254, 'eval_precision_class_0': 0.9510869565217391, 'eval_recall_class_0': 0.9109109109109109, 'eval_f1_class_0': 0.9305654974946314, 'eval_precision_class_1': 0.9146855828220859, 'eval_recall_class_1': 0.9532467532467532, 'eval_f1_class_1': 0.9335681440172194, 'eval_runtime': 37.5597, 'eval_samples_per_second': 266.243, 'eval_steps_per_second': 8.333, 'epoch': 3.0}


## Baseline 5 — BERT + DAPT + IMDB Fine-tune

In [ ]:
# =========================
# Baseline 5 (Step 1): DAPT — continue MLM pretraining on Pool A
# =========================
dapt_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

dapt_training_args = TrainingArguments(
    output_dir=DAPT_OUTPUT_DIR,
    eval_strategy="no",          # no labelled eval during MLM
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=1e-4,          # higher LR typical for MLM continued pretraining
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)

dapt_trainer = Trainer(
    model=dapt_model,
    args=dapt_training_args,
    train_dataset=pool_a_tok,
    data_collator=data_collator_mlm,
)

print("===== DAPT: Continued MLM pretraining on Pool A =====")
dapt_trainer.train()
dapt_trainer.save_model(DAPT_OUTPUT_DIR)
tokenizer.save_pretrained(DAPT_OUTPUT_DIR)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


===== DAPT: Continued MLM pretraining on Pool A =====


Step,Training Loss
1875,2.438355
3750,2.231703
5625,2.082704


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./bert_dapt/tokenizer_config.json', './bert_dapt/tokenizer.json')

In [ ]:
# =========================
# Baseline 5 (Step 2): Fine-tune DAPT model on IMDB
# =========================
dapt_imdb_model = AutoModelForSequenceClassification.from_pretrained(DAPT_OUTPUT_DIR, num_labels=NUM_LABELS)

dapt_imdb_trainer = Trainer(
    model=dapt_imdb_model,
    args=build_training_args(DAPT_IMDB_OUTPUT_DIR, lr=2e-5, epochs=3),
    train_dataset=imdb_train_tok,
    eval_dataset=imdb_valid_tok,
    data_collator=data_collator_clf,
    compute_metrics=compute_metrics
)

print("===== Baseline 5: BERT + DAPT + IMDB fine-tune =====")
dapt_imdb_trainer.train()
dapt_imdb_trainer.save_model(DAPT_IMDB_OUTPUT_DIR)
tokenizer.save_pretrained(DAPT_IMDB_OUTPUT_DIR)

baseline_results.append(evaluate_on_pool_c(dapt_imdb_trainer, pool_c_tok, "5. BERT + DAPT + IMDB"))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ./bert_dapt
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


===== Baseline 5: BERT + DAPT + IMDB fine-tune =====


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Precision Class 0,Recall Class 0,F1 Class 0,Precision Class 1,Recall Class 1,F1 Class 1
1,0.248539,0.222812,0.921900,0.921879,0.936298,0.905400,0.920590,0.908422,0.938400,0.923168
2,0.141577,0.258742,0.923000,0.922956,0.903780,0.946800,0.924790,0.944141,0.899200,0.921123
3,0.072484,0.317781,0.928400,0.928399,0.932029,0.924200,0.928098,0.924831,0.932600,0.928699


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Evaluating: 5. BERT + DAPT + IMDB ---


{'eval_loss': 0.2562616765499115, 'eval_accuracy': 0.9317, 'eval_macro_f1': 0.9316038178688781, 'eval_precision_class_0': 0.965658747300216, 'eval_recall_class_0': 0.8950950950950951, 'eval_f1_class_0': 0.929038961038961, 'eval_precision_class_1': 0.9024208566108007, 'eval_recall_class_1': 0.9682317682317683, 'eval_f1_class_1': 0.9341686746987952, 'eval_runtime': 37.5822, 'eval_samples_per_second': 266.083, 'eval_steps_per_second': 8.328, 'epoch': 3.0}


## Baseline 6 — Full Pipeline: BERT + DAPT + IMDB + Pseudo-labeling

In [ ]:
# =========================
# Baseline 6: Pseudo-labeling using DAPT+IMDB teacher
# =========================
pseudo_b6_tok = generate_pseudo_labels(dapt_imdb_trainer, pool_b_tok)

pseudo_b6_model = AutoModelForSequenceClassification.from_pretrained(DAPT_IMDB_OUTPUT_DIR, num_labels=NUM_LABELS)

pseudo_b6_trainer = Trainer(
    model=pseudo_b6_model,
    args=build_training_args(PSEUDO_DAPT_OUTPUT_DIR, lr=1e-5, epochs=3, train_batch=8),
    train_dataset=pseudo_b6_tok,
    eval_dataset=pool_c_tok,
    data_collator=data_collator_clf,
    compute_metrics=compute_metrics
)

print("===== Baseline 6: BERT + DAPT + IMDB + Pseudo-labeling (Full Pipeline) =====")
pseudo_b6_trainer.train()
pseudo_b6_trainer.save_model(PSEUDO_DAPT_OUTPUT_DIR)

baseline_results.append(evaluate_on_pool_c(pseudo_b6_trainer, pool_c_tok, "6. BERT + DAPT + IMDB + Pseudo (Full)"))

Pseudo-labeled samples kept: 9411 / 10000 (threshold=0.9)


Map:   0%|          | 0/9411 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

===== Baseline 6: BERT + DAPT + IMDB + Pseudo-labeling (Full Pipeline) =====


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Precision Class 0,Recall Class 0,F1 Class 0,Precision Class 1,Recall Class 1,F1 Class 1
1,0.056613,0.454740,0.933100,0.933006,0.967171,0.896496,0.930494,0.903724,0.969630,0.935518
2,0.020841,0.507664,0.937900,0.937843,0.965121,0.908509,0.935960,0.913741,0.967233,0.939726
3,0.006452,0.555520,0.936000,0.935916,0.968986,0.900701,0.933596,0.907411,0.971229,0.938236


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Evaluating: 6. BERT + DAPT + IMDB + Pseudo (Full) ---


{'eval_loss': 0.5078960061073303, 'eval_accuracy': 0.9379, 'eval_macro_f1': 0.9378429342194776, 'eval_precision_class_0': 0.9651212250106338, 'eval_recall_class_0': 0.9085085085085085, 'eval_f1_class_0': 0.9359595751263278, 'eval_precision_class_1': 0.9137410343525859, 'eval_recall_class_1': 0.9672327672327672, 'eval_f1_class_1': 0.9397262933126274, 'eval_runtime': 37.6342, 'eval_samples_per_second': 265.716, 'eval_steps_per_second': 8.317, 'epoch': 3.0}


## Final Comparison Table

In [ ]:
# =========================
# Final comparison table
# =========================
summary = pd.DataFrame([
    {
        "Baseline": r["baseline"],
        "Accuracy": round(r["eval_accuracy"], 4),
        "Macro F1": round(r["eval_macro_f1"], 4),
    }
    for r in baseline_results
])

print("\n===== Final Results on Pool C (held-out Amazon test set) =====")
print(summary.to_string(index=False))


===== Final Results on Pool C (held-out Amazon test set) =====
                              Baseline  Accuracy  Macro F1
                              1. VADER    0.7247    0.7128
                        2. BERT + IMDB    0.9138    0.9136
3. BERT + IMDB + Small Labelled Amazon    0.9440    0.9440
     4. BERT + IMDB + Pseudo (no DAPT)    0.9321    0.9321
                 5. BERT + DAPT + IMDB    0.9317    0.9316
 6. BERT + DAPT + IMDB + Pseudo (Full)    0.9379    0.9378
